<a href="https://colab.research.google.com/github/alexklupsch/wur_deep_learning/blob/main/single_label_resnet18.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Single Label Image Classification -- UCM airborne images -- Resnet18 Fine-Tuning

This notebook shows an easy implementation of the fine-tuning of a resnet18 model to classify the 21 classes of the UCM dataset showing airborne Land Use categories.
It follows the implementation of a medium article (https://medium.com/@imabhi1216/fine-tuning-a-pre-trained-resnet-18-model-for-image-classification-on-custom-dataset-with-pytorch-02df12e83c2c)
and adds wandb (weights and biases to the project).


## 1. Data Loading and Preparation

In [2]:
import os
import zipfile

## loading the data and directory handling
! git clone https://git.wur.nl/lobry001/ucmdata.git
os.chdir('ucmdata')

with zipfile.ZipFile('UCMerced_LandUse.zip', 'r') as zip_ref:
  zip_ref.extractall('UCMImages')

!mv UCMImages/UCMerced_LandUse/Images .
!rm -rf UCMImages README.md UCMerced_LandUse.zip
!ls

UCM_images_path = "Images/"
Multilabels_path = "LandUse_Multilabeled.txt"

Cloning into 'ucmdata'...
remote: Enumerating objects: 8, done.
remote: Total 8 (delta 0), reused 0 (delta 0), pack-reused 8 (from 1)
Receiving objects: 100% (8/8), 316.99 MiB | 16.05 MiB/s, done.
Images	LandUse_Multilabeled.txt


In [4]:
## Accessing the images
dir = "/content/ucmdata/Images"
image_dict = {} # dict for key:class, value: [image_paths]
class_index_dict = {} # dict for key:class_name, value: class_num

for class_num, class_name in enumerate(os.listdir(dir)):
  class_index_dict[class_name] = class_num
  class_dir = os.path.join(dir, class_name)

  if os.path.isdir(class_dir): # sort out *.txt-file
    image_paths = [os.path.join(class_dir, image) for image in os.listdir(class_dir)]
    image_dict[class_num] = image_paths


In [ ]:
import random

def split_train_test(image_dict, test_ratio):
  random.seed(42) # to ensure the same test set for different models
  train_dataset = []
  test_dataset = []

  for label, image_paths in image_dict.items():
    random.shuffle(image_paths)

    split_index = int(len(image_paths) * (1- test_ratio))

    train_dataset.extend([(image_path, label) for image_path in image_paths[:split_index]])
    test_dataset.extend([(image_path, label) for image_path in image_paths[split_index:]])

  return train_dataset, test_dataset

train_dataset, test_dataset = split_train_test(image_dict, test_ratio=0.15)